# 共通コード
次のセルが実習で使う共通の関数などのコードになります。
次のセルを実行しておきましょう。

In [ ]:
# 共通コード
# このセルを実行してください
from PIL import Image
from IPython.display import display
from IPython.display import clear_output
import cv2
import numpy as np
from pathlib import Path

def convert_pil(im, bgr=True):
    if isinstance(im, np.ndarray):
        if bgr:
            im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        im = Image.fromarray(im)
    if isinstance(im, Image.Image):
        pass
    if isinstance(im, str):
        im = Image.open(im)
    if isinstance(im, Image.Image):
        pass
    else:
        raise ValueError("Invalid data type.")

    return im

# 画像を表示（PIL or numpy array(opencv) or file path）
def show_image(im, bgr=True, handle=None):
    im = convert_pil(im)

    if handle is None:
        display(im)
    else:
        handle.update(im)

# 複数の画像を表示（PIL or numpy array(opencv) or file path）
# titlesにいれた文字列をタイトルとして表示
# titlesの値がNoneの場合はfile pathの場合はファイル名を表示、それ以外（PIL or numpy arrya）の場合はタイトルを表示しない。   
def show_images(im_list, titles=[], bgr=True):    
    titles += [None] * len(im_list)
    for im, title in zip(im_list, titles):
        if title is None:
            if isinstance(im, str):
                title = Path(im).name
            else:
                title = ""
            
        print(title)
        show_image(im, bgr=bgr)

# 指定ディレクトリの画像一覧を取得する
def get_image_path_list(root_path, recursive=False, exts=None, pathlib=False):
    root_path = Path(root_path)
    
    if exts is None:
        exts = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp'}
    else:
        exts = {e.lower() if e.startswith('.') else f'.{e.lower()}' for e in exts}
   
    if recursive == True:
        image_path_list = [f for f in root_path.rglob('*') if f.is_file() and f.suffix.lower() in exts]
    else:
        image_path_list = [f for f in root_path.iterdir() if f.is_file() and f.suffix.lower() in exts]

    if not pathlib:
        image_path_list = [str(p) for p in image_path_list]

    return image_path_list

<hr>

# 実運用を想定した推論コード
実際の運用ではカメラシステムなどの制御も含んだソフトウェアで異常検知モデルも運用することになります。
Engineクラスでも推論ができますが、入力データ指定の自由度やリアルタイム性から、直接モデルをつかう形のほうが適しています。

下記はモデル設定ファイル（cfgファイル）とモデルファイル（cpktファイル）をつかって、推論を行うPythonサンプルコードです。

ONNXやPyTorchなどをつかってアプリケーションにモデルを組み込んでいる方にとっては馴染みのある形だと思います。

下記サンプルではAnomalibで学習で使用・生成された以下２つのファイルがあれば推論処理が可能となります。

* モデル設定ファイル（cfgファイル）
* モデルファイル（cpktファイル）

In [ ]:
from anomalib.models import get_model
from omegaconf import OmegaConf
import torch
from torchvision import transforms

import cv2

model_path = "outputs/Patchcore/wood/latest/weights/lightning/model.ckpt"
model_cfg_path = "patchcore.yaml"

#data_path = "datasets/wood/test/OK/IMG_3796_0002.png"
data_path = "datasets/wood/test/NG/IMG_3792_0200.png"


cfg = OmegaConf.load(model_cfg_path)
model = get_model(cfg.model)
ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["state_dict"], strict=False)

model.eval()

# 
im = cv2.imread(data_path)
im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

transform = transforms.Compose([
    transforms.ToPILImage(),        # numpy → PIL
    transforms.Resize((256, 256)),  # resize
    transforms.ToTensor(),          # 0-1 + HWC→CHW
])

x = transform(im)
x = x.unsqueeze(0)

with torch.no_grad():
    pred = model(x)

print("score", pred.pred_score.item())
print("pred label", pred.pred_label.item())
print("anomaly_map", pred.anomaly_map.shape)
print("mask", pred.pred_mask.shape)

# 可視化コード
Anomalibの機能でもヒートマップ画像など、可視化画像の生成は可能ですが、取り回しのよい形で可視化ライブラリを用意しました。

In [ ]:
import cv2
import numpy as np
import logging

logger = logging.getLogger()

def generate_heatmap(anomaly_map, normalize=True, cut_th=None, bgr=True):
    anomaly_map = anomaly_map.squeeze()
    
    if cut_th is None:
        mask = None
    else:
        mask = anomaly_map < cut_th

    if normalize:
        anomaly_map = (anomaly_map - anomaly_map.min()) / (anomaly_map.max() - anomaly_map.min() + 1e-8)
    
    anomaly_map = (anomaly_map * 255).astype(np.uint8)

    heatmap = cv2.applyColorMap(anomaly_map, cv2.COLORMAP_JET)

    if bgr:
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2BGRA)
    else:
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_RGB2BGRA)

    return heatmap, mask

def overlay_heatmap(im_base, anomaly_map, normalize=True, alpha=0.5, cut_th=None, bgr=True):
    im_heatmap, mask = generate_heatmap(anomaly_map, normalize, cut_th, bgr)
    im_overlay = overlay(im_base, im_heatmap, alpha, mask)

    return im_overlay

def overlay(im_base, im_overlay, alpha=0.5, mask=None):
    h, w = im_base.shape[:2]

    im_overlay = cv2.resize(im_overlay, (w, h))
    im_overlay_rgb = im_overlay[:, :, :3]
    im_alpha = (im_overlay[:, :, 3:4] / 255.0) * alpha

    if mask is not None:
        mask = cv2.resize(mask.astype(np.uint8), (w, h)).astype(bool)
        im_alpha[mask] = 0.0

    im_overlay= (im_overlay_rgb * im_alpha + im_base * (1 - im_alpha)).astype("uint8")
    
    return im_overlay

def generate_mask(anomaly_map, threshold=0.5, kernel_size=4):
    anomaly_map = anomaly_map.squeeze()
    mask = np.zeros_like(anomaly_map, dtype=np.uint8)
    mask[anomaly_map > threshold] = 255

    l = np.arange(-kernel_size, kernel_size + 1)
    x, y = np.meshgrid(l, l)
    kernel = np.array((x**2 + y**2) <= kernel_size**2, dtype=np.uint8)

    im_opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    return im_opened

def overlay_mask_edge(
        im_base,
        anomaly_map,
        threshold=0.5,
        kernel_size=4,
        color=(0, 0, 255),
        backcolor=None,
        thickness=1,
        anti_aliasing=True,
    ):
    h, w = im_base.shape[:2]

    anomaly_map = anomaly_map.squeeze()

    mask = generate_mask(anomaly_map, threshold, kernel_size)
    mask = cv2.resize(mask, (w, h))

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    im_overlay = im_base.copy()

    line_type = cv2.LINE_AA if anti_aliasing else cv2.LINE_8

    if backcolor is not None:
        cv2.drawContours(im_overlay, contours, -1, backcolor, thickness=thickness+1, lineType=line_type)
    cv2.drawContours(im_overlay, contours, -1, color, thickness=thickness, lineType=line_type)

    return im_overlay


## 可視化を実行

In [ ]:
# 先程の推論結果
print("score", pred.pred_score.item())
print("pred label", pred.pred_label.item())
print("anomaly_map", pred.anomaly_map.shape)
print("mask", pred.pred_mask.shape)

In [ ]:
# 元画像と異常マップヒートマップを合成した画像を生成
# 色が赤くなるほど異常度が高い
anomaly_map = pred.anomaly_map.detach().cpu().numpy()
im_heatmap = overlay_heatmap(im, anomaly_map)

show_image(im_heatmap)

In [ ]:
# 異常スコア0.5以下は表示カット
anomaly_map = pred.anomaly_map.detach().cpu().numpy()
im_heatmap_cut = overlay_heatmap(im, anomaly_map, cut_th=0.5)

show_image(im_heatmap_cut)

In [ ]:
# 異常箇所の輪郭を描画
anomaly_map = pred.anomaly_map.detach().cpu().numpy()
im_mask_edge = overlay_mask_edge(im, anomaly_map)

show_image(im_mask_edge)

In [ ]:
# ヒートマップ描画の際の標準化をOFF
# デフォルトで標準化ONになっている。標準化で相対的な差で表現し認識しやすくしている。
# ※標準化すると絶対値に意味がなくなるのでそこは注意
anomaly_map = pred.anomaly_map.detach().cpu().numpy()
im_heatmap_norm_off = overlay_heatmap(im, anomaly_map, normalize=False)

show_images([im_heatmap, im_heatmap_norm_off], titles=["Normalize:ON", "Normalize:OFF"], bgr=True)

# Anomalib学習済モデルの使える簡易外観検査アプリ

Anomalib学習済モデルの使える簡易外観検査アプリを公開しています。
<br>
https://github.com/cm-koga/anomalib-compact-inspection-system
Apache-2.0 license

* 上記の推論のしくみを使っています
* 可視化も上記のものを使っています
* Webカメラで撮影したカメラライブ画像に対して異常検知をリアルタイムで実施し、結果表示をします
* データセット作成のために画像保存できる機能付き

![img](./img/cap_app.png)